In [ ]:
import json
import uuid
import re
import pandas as pd

from snowflake.connector.pandas_tools import write_pandas
from snowflake.snowpark.context import get_active_session

def print_info(msg):    print(f"INFO:    {msg}")
def print_warning(msg): print(f"WARNING: {msg}")
def print_error(msg):   print(f"ERROR:   {msg}")

session = get_active_session()


In [ ]:
staged_files = session.sql("""
    SELECT RELATIVE_PATH
    FROM DIRECTORY(@PERMAFROST_POC.INGEST.RAW_DOCUMENTS_STAGE)
    WHERE RELATIVE_PATH LIKE 'config/%.json'
""").collect()

doc_type_files = [r["RELATIVE_PATH"].split("/")[-1] for r in staged_files]
print_info(f"Found {len(doc_type_files)} config file(s): {doc_type_files}")

if not doc_type_files:
    print_warning("No config JSON files found in stage. Upload first")

In [ ]:
processed_doc_types = []

for filename in doc_type_files:
    try:
        raw = session.sql(f"""
            SELECT $1::VARCHAR AS JSON_TEXT
            FROM @PERMAFROST_POC.INGEST.RAW_DOCUMENTS_STAGE/config/{filename}
            (FILE_FORMAT => PERMAFROST_POC.CONFIG.JSON_RAW_FORMAT)
        """).collect()

        json_text  = "\n".join([row["JSON_TEXT"] for row in raw])
        doc_config = json.loads(json_text)
        doc_type   = doc_config["doc_type"]

        # Defaults for optional keys — won't overwrite if present in JSON
        display_name    = doc_config.get(
            "display_name",
            doc_type.replace("_", " ").title()
        )
        schema_version  = doc_config.get("schema_version", "1.0")
        mandatory_fields = doc_config.get("mandatory_fields", [])
        optional_fields = doc_config.get("optional_fields", [])
        synonyms        = doc_config.get("synonyms", {})
        thresholds      = doc_config.get("confidence_thresholds", {
            "auto_approve": 0.85,
            "review_below": 0.85
        })
        weights         = doc_config.get("confidence_weights", {
            "field_completeness": 0.50,
            "extract_certainty":  0.50
        })
        dedup_key_fields = doc_config.get("dedup_key_fields", None)
        prompt_template  = doc_config.get("prompt_template", None)
        llm_model        = doc_config.get("llm_model", None)

        def sql_json_or_null(val):
            if val is None:
                return "NULL"
            return f"PARSE_JSON($${json.dumps(val)}$$)"

        def sql_str_or_null(val):
            if val is None:
                return "NULL"
            escaped = val.replace("'", "''")
            return f"'{escaped}'"

        session.sql(f"""
            MERGE INTO PERMAFROST_POC.CONFIG.DOC_TYPE_CONFIG AS target
            USING (
                SELECT
                    '{doc_type}'                                          AS DOC_TYPE,
                    '{display_name}'                                      AS DOC_DISPLAY_NAME,
                    '{schema_version}'                                    AS SCHEMA_VERSION,
                    PARSE_JSON($${json.dumps(mandatory_fields)}$$)        AS MANDATORY_FIELDS,
                    PARSE_JSON($${json.dumps(optional_fields)}$$)         AS OPTIONAL_FIELDS,
                    {sql_json_or_null(synonyms if synonyms else None)}    AS SYNONYMS,
                    PARSE_JSON($${json.dumps(thresholds)}$$)              AS CONFIDENCE_THRESHOLDS,
                    PARSE_JSON($${json.dumps(weights)}$$)                 AS CONFIDENCE_WEIGHTS,
                    {sql_json_or_null(dedup_key_fields)}                  AS DEDUP_KEY_FIELDS,
                    {sql_str_or_null(prompt_template)}                    AS PROMPT_TEMPLATE,
                    {sql_str_or_null(llm_model)}                          AS LLM_MODEL,
                    TRUE                                                  AS IS_ACTIVE
            ) AS source ON target.DOC_TYPE = source.DOC_TYPE
            WHEN MATCHED THEN UPDATE SET
                DOC_DISPLAY_NAME      = source.DOC_DISPLAY_NAME,
                SCHEMA_VERSION        = source.SCHEMA_VERSION,
                MANDATORY_FIELDS      = source.MANDATORY_FIELDS,
                OPTIONAL_FIELDS       = source.OPTIONAL_FIELDS,
                SYNONYMS              = source.SYNONYMS,
                CONFIDENCE_THRESHOLDS = source.CONFIDENCE_THRESHOLDS,
                CONFIDENCE_WEIGHTS    = source.CONFIDENCE_WEIGHTS,
                DEDUP_KEY_FIELDS      = source.DEDUP_KEY_FIELDS,
                PROMPT_TEMPLATE       = source.PROMPT_TEMPLATE,
                LLM_MODEL             = source.LLM_MODEL,
                IS_ACTIVE             = source.IS_ACTIVE,
                UPDATED_AT            = CURRENT_TIMESTAMP()
            WHEN NOT MATCHED THEN INSERT (
                DOC_TYPE, DOC_DISPLAY_NAME, SCHEMA_VERSION,
                MANDATORY_FIELDS, OPTIONAL_FIELDS, SYNONYMS,
                CONFIDENCE_THRESHOLDS, CONFIDENCE_WEIGHTS, DEDUP_KEY_FIELDS,
                PROMPT_TEMPLATE, LLM_MODEL, IS_ACTIVE
            ) VALUES (
                source.DOC_TYPE, source.DOC_DISPLAY_NAME, source.SCHEMA_VERSION,
                source.MANDATORY_FIELDS, source.OPTIONAL_FIELDS, source.SYNONYMS,
                source.CONFIDENCE_THRESHOLDS, source.CONFIDENCE_WEIGHTS,
                source.DEDUP_KEY_FIELDS, source.PROMPT_TEMPLATE,
                source.LLM_MODEL, source.IS_ACTIVE
            )
        """).collect()

        print_info(f"Upserted DOC_TYPE_CONFIG: {doc_type}")
        processed_doc_types.append((filename, doc_type, doc_config))

    except Exception as e:
        print_error(f"Failed to process {filename}: {e}")


In [ ]:
classify_prompt = """You are analyzing a multi-page file that may contain multiple documents.
Your job is to identify each distinct document — by both its type AND its
boundaries. Two pages of the same document type are only the same segment
if they are part of the same physical document (e.g. page 2 continues
page 1 of the same certificate). If a new document of the same type starts
(new certificate number, new issuing authority, new header, new reference),
it must be a separate segment.

Document types:
- health_certificate
- commercial_invoice
- packing_list
- bill_of_lading
- catch_certificate
- country_of_origin_certificate
- unknown

Rules:
- Each distinct physical document = one segment, even if same type as adjacent segment
- Look for new document headers, certificate numbers, reference numbers,
  issuing authority names, or layout resets as boundary signals
- A continuation page (e.g. page 2 of a long invoice) belongs to the same segment
- If uncertain whether two same-type pages are one doc or two, prefer splitting

Return ONLY valid JSON, no explanation, no markdown:
{{
  "segments": [
    {{
      "doc_type": "<doc_type>",
      "page_start": <1-based int>,
      "page_end": <1-based int>,
      "confidence": <float 0.0-1.0>,
      "boundary_signal": "<brief reason why a new segment starts here>"
    }}
  ]
}}

Document pages:
{pages}"""  

config_rows = [
    {
        'CONFIG_KEY':   'classify_model',
        'CONFIG_VALUE': 'claude-sonnet-4-6',
        'DESCRIPTION':  'Model used for document classification via AI_COMPLETE',
    },
    {
        'CONFIG_KEY':   'classify_prompt',
        'CONFIG_VALUE': classify_prompt,
        'DESCRIPTION':  'Prompt template for classification — {pages} is the only placeholder',
    },
]

s.write_pandas(
    pd.DataFrame(config_rows),
    table_name='PIPELINE_CONFIG',
    database='PERMAFROST_POC',
    schema='CONFIG',
    overwrite=False,
)

info("Seeded PIPELINE_CONFIG")